In [1]:
import numpy as np
import scipy.optimize as opt
bisect = opt.bisect
import pandas as pd
import copy
from matplotlib import pyplot as plt

import time

In [2]:

def compute_force(speed, objectArea, dragCoefficient, mediumDensity):
    return 0.5 * dragCoefficient * mediumDensity * (objectArea/2) * (speed * speed)

def apply_water_current_force(velocity, areaBall, CdWater, rhoWater, waterSpeed, massBall, dt):
    direction = 1.0 if waterSpeed >= 0.0 else -1.0

    water_force = compute_force(abs(waterSpeed), areaBall, CdWater, rhoWater)
    total_force = 0.5 * water_force

    acc = total_force / massBall
    delta_speed = acc * dt

    # Adds to X velocity based on water current direction
    velocity[0] += delta_speed * direction

    return velocity

def calculate_deceleration(velocity, areaBall, CdWater, rhoWater, massBall, dt):
    velocity_magnitude = np.linalg.norm(velocity)

    if velocity_magnitude > 0: # i.e., ball has not stopped

        # drag forces
        drag_water = compute_force(velocity_magnitude, areaBall, CdWater, rhoWater)

        # Drag applied to half of the ball area
        total_drag_force = 0.5 * drag_water

        deceleration = total_drag_force / massBall
        delta_speed = deceleration * dt

        new_speed = max(velocity_magnitude - delta_speed, 0.0)

        # normalized velocity (scales len to 1.0)
        velocity = (velocity / velocity_magnitude) * new_speed

    return velocity


def pointDistance(p1, p2):
    return np.sqrt(np.sum((np.array(p1) - np.array(p2))**2))

def getTrajectory(speed,
                  angle,
                  target,
                  water_speed,
                  output = 'trajectory', # can also be 'mindist' to just return 1 number
                  constants = { 'CdWater'      : 0.47,
                                'rhoWater'     : 1000,
                                'massBall'     : 1.0,
                                'radiusBall'   : 0.049999995,
                                'radius'       : 0.0,
                                'dt'           : 1.0 / 72}
                  ):

    # Constants
    CdWater = constants['CdWater']
    rhoWater = constants['rhoWater']
    massBall = constants['massBall']
    radiusBall = constants['radiusBall']
    radius = constants['radius']
    dt = constants['dt']

    areaBall = np.pi * (radiusBall**2)    # incorrect, but this was used in the task...

    speed = np.ravel(speed)[0]

    # translate to velocity vector for the ball:
    angle_rad = np.radians(angle)
    velocity = np.array([speed * np.cos(angle_rad), 0.0, speed * np.sin(angle_rad)])

    # starting position of the ball:
    position = np.array([0.0, 0.0, 0.0])

    prevdist = pointDistance(target, position)
    trajectory = np.reshape( np.concatenate((copy.deepcopy(position)[[0,2]],np.array([prevdist]) )), (1,3))
    t = 0

    ongoing = True

    # uses EUCLIDEAN distance now:
    while ongoing:

        # print('- - time:'+str(t))
        # 1. ApplyWaterCurrentForce()
        velocity = apply_water_current_force(velocity, areaBall, CdWater, rhoWater, water_speed, massBall, dt)

        # 2. Update Position
        position += velocity * dt

        # 3. CalculateDeceleration()
        # this check is done in the function, so not repeating it here:
        # if np.linalg.norm(velocity) > 0.0: # if ball is moving (i.e., has a velocity)...
        velocity = calculate_deceleration(velocity, areaBall, CdWater, rhoWater, massBall, dt)


        t += dt

        dist = pointDistance(position, target)
        newrow = np.reshape( np.concatenate((copy.deepcopy(position)[[0,2]],np.array([dist]) )), (1,3))
        trajectory = np.concatenate((trajectory, newrow))

        # # when to stop? far beyond target_Z
        # if (position[2]/(5.0)) > target[2]:
        #     ongoing = False

        # if water_speed != 0:
        #     if (np.sign(water_speed) - np.sign(velocity[0])) >= 0:
        #         if np.sign(position[0] - target[0]) == np.sign(water_speed):
        #             ongoing = False
        # else:
        #     # or if the ball stopped moving (only applies when water speed is 0)
        #     if np.linalg.norm(velocity) < 0.0000001:
        #         ongoing = False

        # if water_speed == 0:
        #     if np.linalg.norm(velocity) < 0.0000001:
        #         ongoing = False
        #     if pointDistance(position, [0,0,0]) > (target_dist * 1.10):
        #         ongoing = False
        # else:
        #     if position[2] > target[2]:
        #         if dist > prevdist:
        #             ongoing = False

        # prevdist = dist

        if pointDistance(position, [0,0,0]) > 3:
            ongoing = False

    # print(trajectory)


    if output == 'trajectory':
        return trajectory
    elif output == 'mindist':
        # print( np.min(trajectory[:,2]) )
        return np.abs( np.min(trajectory[:,2])-radius )
    else:
        return None

In [3]:

def getSolutions(water_speed, target):

    constants_mid = {'CdWater'      : 0.47,
                     'rhoWater'     : 1000,
                     'massBall'     : 1.0,
                     'radiusBall'   : 0.049999995,
                     'radius'       : 0,
                     'dt'           : 1.0 / 72}

    constants_edge =  {'CdWater'      : 0.47,
                     'rhoWater'     : 1000,
                     'massBall'     : 1.0,
                     'radiusBall'   : 0.049999995,
                     'radius'       : 0.175,
                     'dt'           : 1.0 / 72}

    angles     = []
    speeds_mid = []
    speeds_lo  = []
    speeds_hi  = []

    target_angle = np.arctan2(target[2], target[0])
    target_angle = int(np.floor(np.degrees(target_angle)))

    pd = pointDistance(target, [0,0,0])
    additional_angle = np.arctan(0.175/pd)

    target_angle += additional_angle

    for angle in np.linspace(20, target_angle, (int(np.floor((target_angle-20)/.25))+1)):

        angles.append(angle)

        m = opt.minimize_scalar(getTrajectory,
                                bounds = [0.5,8],
                                args=(angle, target, water_speed, 'mindist', constants_mid),
                                method='bounded',
                                tol=10e-4
                               )

        if m.fun < .015:
            speeds_mid.append(m.x)

            m_lo = opt.minimize_scalar(getTrajectory,
                                    bounds = [0.5,m.x],
                                    args=(angle, target, water_speed, 'mindist', constants_edge),
                                    method='bounded',
                                    tol=10e-4
                                )
            speeds_lo.append(m_lo.x if m_lo.fun < .01 else np.nan)
            m_hi = opt.minimize_scalar(getTrajectory,
                                    bounds = [m.x,9],
                                    args=(angle, target, water_speed, 'mindist', constants_edge),
                                    method='bounded',
                                    tol=10e-4
                                )
            speeds_hi.append(m_hi.x if m_hi.fun < .01 else np.nan)
        else:
            speeds_mid.append(np.nan)
            speeds_lo.append(np.nan)
            speeds_hi.append(np.nan)

        # angles.append(angle)
        # speeds_mid.append(m.x if m.fun < .01 else np.nan)
        # speeds_lo.append(m_lo.x if m_lo.fun < .01 else np.nan)
        # speeds_hi.append(m_hi.x if m_hi.fun < .01 else np.nan)


    return {'angles'     : angles,
            'speeds_mid' : speeds_mid,
            'speeds_lo'  : speeds_lo,
            'speeds_hi'  : speeds_hi}

In [81]:
water_speeds = [-2, -3]

# water_speeds = [-3]
targets = {
            'L60' : [-.6, 0, 1.4],
            'L30' : [-.3, 0, 1.4],
            'R30' : [ .3, 0, 1.4],
            'R60' : [ .6, 0, 1.4]   }

speeds = np.linspace(0,8,81)
angles = np.linspace(10,170,641)

output_path = "C:/Users/jacob/water_current_MA/data/solution_space"


for water_speed_idx in range(len(water_speeds)):
    water_speed = water_speeds[water_speed_idx]
    for target_name in targets.keys():
        target = targets[target_name]

        start_time = time.time()


        errors = np.full((len(speeds),len(angles)), np.nan)

        constants =  {      'CdWater'      : 0.47,
                            'rhoWater'     : 1000,
                            'massBall'     : 1.0,
                            'radiusBall'   : 0.049999995,
                            'radius'       : 0,
                            'dt'           : 1.0 / 72          }

        for v_idx in range(len(speeds)):
            for a_idx in range(len(angles)):
                errors[v_idx,a_idx] = getTrajectory(speed = speeds[v_idx],
                                                    angle = angles[a_idx],
                                                    target = target,
                                                    water_speed = water_speed,
                                                    output = 'mindist',
                                                    constants = constants)


        errdf = pd.DataFrame(errors)
        errdf.columns = angles
        errdf.index = speeds
        
        filename = 'errors_'+str(water_speed)+'_'+target_name+'.csv'
        errdf.to_csv(f'{output_path}/{filename}')

        print('Written file: '+filename)

        end_time = time.time()
        duration = end_time - start_time
        # display simulation and file save duration 
        print(duration)


Written file: errors_-2_L60.csv
315.5654630661011
Written file: errors_-2_L30.csv
324.7653431892395
Written file: errors_-2_R30.csv
318.9845516681671
Written file: errors_-2_R60.csv
321.633900642395
Written file: errors_-3_L60.csv
238.1599042415619
Written file: errors_-3_L30.csv
239.50514388084412
Written file: errors_-3_R30.csv
241.38315153121948
Written file: errors_-3_R60.csv
240.4496042728424


In [79]:
np.linspace(0,8,50)

def check_simulation_resolution(valMin=0, valMax=8, itr=50):

    difference_sum = 0

    v = np.linspace(valMin, valMax, itr)

    for i in range(len(v)):

        # print(i)

        current_item = v[i]

        if i > 0:
            previous_item = v[i-1]
        else:
            previous_item = current_item


        # print("Current:", current_item)
        # print("Prev:", previous_item)

        # find difference between previous and current
        difference = current_item - previous_item
        # print(difference)

        difference_sum += difference

    print(difference_sum / len(v))

print(check_simulation_resolution(10,170,641))
print(check_simulation_resolution(0,8,81))

0.24960998439937598
None
0.1
None
